<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 05 — Silver: MEN (Estadísticas Educación Municipal)

Indicadores educativos a nivel municipio-año (cobertura, deserción, aprobación, etc.).

## FILTROS APLICADOS

| # | Filtro | Justificación |
|---|---|---|
| F1 | `COD_MPIO IS NOT NULL AND ANO IS NOT NULL` | `(COD_MPIO, ANO)` es la clave compuesta del Silver MEN y la **clave de join con el panel municipal**. Registros sin ambas claves no aportan al pipeline y romperían los joins. |
| F2 | `POBLACION_5_16 > 0` | Municipios sin población infantil reportada (`POBLACION_5_16 ≤ 0` o nulo) corresponden a registros administrativos vacíos del MEN; sus tasas y coberturas son indeterminadas y contaminan los promedios departamentales. |

## TRANSFORMACIONES APLICADAS

| # | Transformación | Justificación |
|---|---|---|
| T1 | **Slugify de columnas**: `unicodedata.normalize('NFD')` + drop combining + `replace(' ','_').upper()` | Los nombres originales tienen acentos (`CÓDIGO`, `POBLACIÓN`, `BÁSICA`) y caracteres especiales (`Ñ`, comas), que rompen `df.select(...)` en Spark. Aplicado a las 41 columnas. |
| T2 | Cast `ANO` → IntegerType ; `lpad(CODIGO_MUNICIPIO, 5)` → `COD_MPIO` ; `lpad(CODIGO_DEPARTAMENTO, 2)` → `COD_DEPTO` | Códigos DANE como string padded son necesarios para join contra ICFES/Internet/SISBEN (donde también están así). |
| T3 | **Parsear porcentajes**: `regexp_replace('%','')` → DoubleType / 100 sobre todas las columnas `TASA_*`, `COBERTURA_*`, `DESERCION*`, `APROBACION*`, `REPROBACION*`, `REPITENCIA*` | Bronze los preserva como strings tipo `"56.11%"`. Sin parsing no se pueden usar como features numéricas en correlaciones, regresión ni K-Means. División por 100 para escala 0..1 (consistente con `pct_internet_icfes`). |
| T4 | Cast `POBLACION_5_16`, `SEDES_CONECTADAS_A_INTERNET` → LongType ; `TAMANO_PROMEDIO_DE_GRUPO` → DoubleType | Enteros y decimales numéricos para agregaciones; sin tipado correcto, `sum/avg` los trata como string. |
| T5 | `partitionBy(ANO)` al escribir Parquet | Acelera queries filtradas por año en Gold y EDA. |

## 1. Setup y carga

In [1]:
import sys, unicodedata
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, LongType

spark = get_spark("Entrega2-Silver-MEN", executor_memory="2g", driver_memory="2g", cores=2)
df = spark.read.parquet(P.BRONZE_PQ_MEN)
print(f"Filas: {df.count():,}   Cols: {len(df.columns)}")
print("Columnas originales:")
for c in df.columns: print("  ", repr(c))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/22 08:16:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Filas: 15,707   Cols: 41
Columnas originales:
   'AÑO'
   'CÓDIGO_MUNICIPIO'
   'MUNICIPIO'
   'CÓDIGO_DEPARTAMENTO'
   'DEPARTAMENTO'
   'CÓDIGO_ETC'
   'ETC'
   'POBLACIÓN_5_16'
   'TASA_MATRICULACIÓN_5_16'
   'COBERTURA_NETA'
   'COBERTURA_NETA_TRANSICIÓN'
   'COBERTURA_NETA_PRIMARIA'
   'COBERTURA_NETA_SECUNDARIA'
   'COBERTURA_NETA_MEDIA'
   'COBERTURA_BRUTA'
   'COBERTURA_BRUTA_TRANSICIÓN'
   'COBERTURA_BRUTA_PRIMARIA'
   'COBERTURA_BRUTA_SECUNDARIA'
   'COBERTURA_BRUTA_MEDIA'
   'TAMAÑO_PROMEDIO_DE_GRUPO'
   'SEDES_CONECTADAS_A_INTERNET'
   'DESERCIÓN'
   'DESERCIÓN_TRANSICIÓN'
   'DESERCIÓN_PRIMARIA'
   'DESERCIÓN_SECUNDARIA'
   'DESERCIÓN_MEDIA'
   'APROBACIÓN'
   'APROBACIÓN_TRANSICIÓN'
   'APROBACIÓN_PRIMARIA'
   'APROBACIÓN_SECUNDARIA'
   'APROBACIÓN_MEDIA'
   'REPROBACIÓN'
   'REPROBACIÓN_TRANSICIÓN'
   'REPROBACIÓN_PRIMARIA'
   'REPROBACIÓN_SECUNDARIA'
   'REPROBACIÓN_MEDIA'
   'REPITENCIA'
   'REPITENCIA_TRANSICIÓN'
   'REPITENCIA_PRIMARIA'
   'REPITENCIA_SECUNDARIA'
   

## 2. Slugify de columnas (sin acentos/espacios, UPPER)

In [2]:
def slugify(name: str) -> str:
    # NFD descompone Á → A + combining acute. Mn = marks, nonspacing.
    no_accent = "".join(ch for ch in unicodedata.normalize("NFD", name) if unicodedata.category(ch) != "Mn")
    return (no_accent.replace(" ", "_")
                     .replace("-", "_")
                     .replace(",", "")
                     .upper())

rename_map = {c: slugify(c) for c in df.columns}
df_r = df
for old, new in rename_map.items():
    df_r = df_r.withColumnRenamed(old, new)
print("Renombres aplicados:", sum(1 for k,v in rename_map.items() if k != v))
print("Columnas nuevas (primeras 15):", df_r.columns[:15])

Renombres aplicados: 25
Columnas nuevas (primeras 15): ['ANO', 'CODIGO_MUNICIPIO', 'MUNICIPIO', 'CODIGO_DEPARTAMENTO', 'DEPARTAMENTO', 'CODIGO_ETC', 'ETC', 'POBLACION_5_16', 'TASA_MATRICULACION_5_16', 'COBERTURA_NETA', 'COBERTURA_NETA_TRANSICION', 'COBERTURA_NETA_PRIMARIA', 'COBERTURA_NETA_SECUNDARIA', 'COBERTURA_NETA_MEDIA', 'COBERTURA_BRUTA']


## 3. Tipificar año y códigos DANE

In [3]:
df_r = (df_r
    .withColumn("ANO", F.col("ANO").cast(IntegerType()))
    .withColumn("CODIGO_MUNICIPIO",    F.lpad(F.col("CODIGO_MUNICIPIO"), 5, "0"))
    .withColumn("CODIGO_DEPARTAMENTO", F.lpad(F.col("CODIGO_DEPARTAMENTO"), 2, "0"))
)
df_r = df_r.withColumnRenamed("CODIGO_MUNICIPIO", "COD_MPIO")
df_r = df_r.withColumnRenamed("CODIGO_DEPARTAMENTO", "COD_DEPTO")
df_r.select("ANO","COD_MPIO","MUNICIPIO","COD_DEPTO","DEPARTAMENTO","POBLACION_5_16").show(5, truncate=False)

+----+--------+-------------+---------+------------+--------------+
|ANO |COD_MPIO|MUNICIPIO    |COD_DEPTO|DEPARTAMENTO|POBLACION_5_16|
+----+--------+-------------+---------+------------+--------------+
|2024|05004   |Abriaquí     |05       |Antioquia   |499           |
|2024|15204   |Cómbita      |15       |Boyacá      |1862          |
|2024|99773   |Cumaribo     |99       |Vichada     |25239         |
|2024|99624   |Santa Rosalía|99       |Vichada     |1157          |
|2024|99524   |La Primavera |99       |Vichada     |2645          |
+----+--------+-------------+---------+------------+--------------+
only showing top 5 rows



## 3b. Aplicación de filtros (F1 + F2)

In [4]:
n0 = df_r.count()
print(f"Filas antes de filtros: {n0:,}")

# F1 — COD_MPIO y ANO no nulos (claves de join con el panel)
df_r = df_r.filter(F.col("COD_MPIO").isNotNull() & F.col("ANO").isNotNull())
n1 = df_r.count()
print(f"  Tras F1 (COD_MPIO y ANO no nulos)        : {n1:>8,}  (eliminó {n0-n1:,})")

# F2 — POBLACION_5_16 > 0 (municipios con población infantil reportada)
df_r = df_r.filter(F.col("POBLACION_5_16").cast(LongType()) > 0)
n2 = df_r.count()
print(f"  Tras F2 (POBLACION_5_16 > 0)             : {n2:>8,}  (eliminó {n1-n2:,})")
print(f"Total eliminado por filtros: {n0-n2:,} ({100*(n0-n2)/n0:.2f}%)")

Filas antes de filtros: 15,707


  Tras F1 (COD_MPIO y ANO no nulos)        :   15,707  (eliminó 0)


  Tras F2 (POBLACION_5_16 > 0)             :   15,700  (eliminó 7)
Total eliminado por filtros: 7 (0.04%)


## 4. Parsear porcentajes  '56.11%' → 0.5611

In [5]:
# Toda columna que contenga '%' en cualquier fila la tratamos como porcentaje
def parse_pct(col):
    cleaned = F.regexp_replace(col, "%", "")
    cleaned = F.when(cleaned == "", None).otherwise(cleaned)
    cleaned = F.when(cleaned == "NULL", None).otherwise(cleaned)
    return (cleaned.cast(DoubleType()) / 100.0)

# Columnas que sabemos que son porcentaje (por el muestreo)
pct_cols = [c for c in df_r.columns if c.startswith("TASA_") or c.startswith("COBERTURA_")
            or c.startswith("DESERCION") or c.startswith("APROBACION")
            or c.startswith("REPROBACION") or c.startswith("REPITENCIA")]
print(f"Columnas porcentaje detectadas: {len(pct_cols)}")
for c in pct_cols:
    df_r = df_r.withColumn(c, parse_pct(F.col(c)))

# POBLACION y TAMAÑO_PROMEDIO_DE_GRUPO y SEDES_CONECTADAS_A_INTERNET son numéricos enteros
for c in ("POBLACION_5_16", "SEDES_CONECTADAS_A_INTERNET"):
    if c in df_r.columns:
        df_r = df_r.withColumn(c, F.col(c).cast(LongType()))
if "TAMANO_PROMEDIO_DE_GRUPO" in df_r.columns:
    df_r = df_r.withColumn("TAMANO_PROMEDIO_DE_GRUPO", F.col("TAMANO_PROMEDIO_DE_GRUPO").cast(DoubleType()))

df_r.printSchema()
df_r.select("ANO","COD_MPIO","COBERTURA_NETA","DESERCION","APROBACION","POBLACION_5_16").show(5)

Columnas porcentaje detectadas: 31


root
 |-- ANO: integer (nullable = true)
 |-- COD_MPIO: string (nullable = true)
 |-- MUNICIPIO: string (nullable = true)
 |-- COD_DEPTO: string (nullable = true)
 |-- DEPARTAMENTO: string (nullable = true)
 |-- CODIGO_ETC: string (nullable = true)
 |-- ETC: string (nullable = true)
 |-- POBLACION_5_16: long (nullable = true)
 |-- TASA_MATRICULACION_5_16: double (nullable = true)
 |-- COBERTURA_NETA: double (nullable = true)
 |-- COBERTURA_NETA_TRANSICION: double (nullable = true)
 |-- COBERTURA_NETA_PRIMARIA: double (nullable = true)
 |-- COBERTURA_NETA_SECUNDARIA: double (nullable = true)
 |-- COBERTURA_NETA_MEDIA: double (nullable = true)
 |-- COBERTURA_BRUTA: double (nullable = true)
 |-- COBERTURA_BRUTA_TRANSICION: double (nullable = true)
 |-- COBERTURA_BRUTA_PRIMARIA: double (nullable = true)
 |-- COBERTURA_BRUTA_SECUNDARIA: double (nullable = true)
 |-- COBERTURA_BRUTA_MEDIA: double (nullable = true)
 |-- TAMANO_PROMEDIO_DE_GRUPO: double (nullable = true)
 |-- SEDES_CONECTADAS_

+----+--------+--------------+--------------------+------------------+--------------+
| ANO|COD_MPIO|COBERTURA_NETA|           DESERCION|        APROBACION|POBLACION_5_16|
+----+--------+--------------+--------------------+------------------+--------------+
|2024|   05004|        0.5611|                 0.0|            0.9968|           499|
|2024|   15204|        0.9533|0.032400000000000005|            0.9414|          1862|
|2024|   99773|         0.507|               0.055|0.7928000000000001|         25239|
|2024|   99624|        0.8142|               0.063|0.8606999999999999|          1157|
|2024|   99524|        0.9096|              0.0516|0.8601000000000001|          2645|
+----+--------+--------------+--------------------+------------------+--------------+
only showing top 5 rows



## 5. Escribir Silver Parquet

In [6]:
import time
t0 = time.time()
(df_r.write
    .mode("overwrite")
    .partitionBy("ANO")
    .option("compression","snappy")
    .parquet(P.SILVER_MEN))
print(f"Escrito en {time.time()-t0:.1f}s")

sv = spark.read.parquet(P.SILVER_MEN)
print(f"Verificación: {sv.count():,} filas")
sv.groupBy("ANO").count().orderBy("ANO").show()

26/05/22 08:17:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Escrito en 6.8s


Verificación: 15,700 filas


+----+-----+
| ANO|count|
+----+-----+
|2011| 1119|
|2012| 1119|
|2013| 1122|
|2014| 1122|
|2015| 1122|
|2016| 1122|
|2017| 1122|
|2018| 1122|
|2019| 1123|
|2020| 1122|
|2021| 1121|
|2022| 1121|
|2023| 1121|
|2024| 1122|
+----+-----+



In [7]:
spark.stop()